# CSV File Checker
Checks for BOM issues, encoding, null columns, duplicates, and blank rows.
Upload a file, run all checks, download the cleaned version.

## 1. Install & Imports

In [ ]:
import pandas as pd
import os
import chardet
from IPython.display import display, FileLink, HTML

try:
    import ipywidgets as widgets
    WIDGETS_AVAILABLE = True
except ImportError:
    WIDGETS_AVAILABLE = False
    print('ipywidgets not installed — using manual file path input instead.')
    print('To enable the upload widget run: pip install ipywidgets')

## 2. Upload File

In [ ]:
if WIDGETS_AVAILABLE:
    uploader = widgets.FileUpload(accept='.csv', multiple=False, description='Upload CSV')
    display(uploader)
else:
    print('Set FILE_PATH below to the full path of your CSV file.')

In [ ]:
# ── Save uploaded file to disk ──────────────────────────────────────────────
# If using the widget, run this cell after uploading.
# If no widget, set FILE_PATH manually and skip this cell.

FILE_PATH = None  # Will be set automatically if widget is used

if WIDGETS_AVAILABLE and uploader.value:
    uploaded_file = list(uploader.value.values())[0]
    FILE_PATH = uploaded_file['metadata']['name']
    with open(FILE_PATH, 'wb') as f:
        f.write(uploaded_file['content'])
    print(f'File saved: {FILE_PATH}')
elif not FILE_PATH:
    FILE_PATH = r'sample_data/test_sample.csv'  # << set manually if no widget
    print(f'Using manual path: {FILE_PATH}')

## 3. Run Checks

In [ ]:
# ── CHECK 1: BOM Detection ───────────────────────────────────────────────────
print('=' * 55)
print('CHECK 1 — BOM Detection')
print('=' * 55)

with open(FILE_PATH, 'rb') as f:
    raw_bytes = f.read(4)

BOM_DETECTED = raw_bytes[:3] == b'\xef\xbb\xbf'

if BOM_DETECTED:
    print('❌ BOM detected (\\ufeff) — file was saved as UTF-8-with-BOM (likely Excel).')
    print('   First column name will have a hidden \\ufeff prefix in MySQL/BQ.')
    print('   Fix: will be applied in the clean step below.')
else:
    print('✅ No BOM detected — file is clean UTF-8 or another encoding.')

In [ ]:
# ── CHECK 2: Encoding Detection ──────────────────────────────────────────────
print('=' * 55)
print('CHECK 2 — Encoding Detection')
print('=' * 55)

with open(FILE_PATH, 'rb') as f:
    detected = chardet.detect(f.read())

print(f'Detected encoding : {detected["encoding"]}')
print(f'Confidence        : {detected["confidence"] * 100:.1f}%')

if detected['encoding'] and detected['encoding'].lower() in ['utf-8-sig', 'utf-8']:
    print('✅ UTF-8 family — safe for BigQuery and MySQL.')
elif detected['encoding'] and detected['encoding'].lower() in ['iso-8859-1', 'latin-1', 'windows-1252']:
    print('⚠️  Latin-1/Windows-1252 detected — may cause issues with special characters in BQ/MySQL.')
    print('   Fix: will re-encode to UTF-8 in the clean step.')
else:
    print(f'⚠️  Unusual encoding: {detected["encoding"]} — review before uploading.')

In [ ]:
# ── Load file for remaining checks ───────────────────────────────────────────
read_encoding = 'utf-8-sig' if BOM_DETECTED else (detected['encoding'] or 'utf-8')
df = pd.read_csv(FILE_PATH, encoding=read_encoding)

print(f'File loaded: {df.shape[0]} rows x {df.shape[1]} columns')
print(f'Columns: {df.columns.tolist()}')

In [ ]:
# ── CHECK 3: Column Name Issues ──────────────────────────────────────────────
print('=' * 55)
print('CHECK 3 — Column Name Issues')
print('=' * 55)

issues = []
for col in df.columns:
    if '\ufeff' in col:
        issues.append(f'  ❌ BOM in column name: "{col}"')
    if col != col.strip():
        issues.append(f'  ❌ Leading/trailing spaces: "{col}"')
    if col == '':
        issues.append(f'  ❌ Empty column name found')

if issues:
    for i in issues:
        print(i)
else:
    print('✅ All column names are clean.')

In [ ]:
# ── CHECK 4: Blank / Empty Rows ──────────────────────────────────────────────
print('=' * 55)
print('CHECK 4 — Blank / Empty Rows')
print('=' * 55)

blank_rows = df[df.isnull().all(axis=1)]
print(f'Fully blank rows : {len(blank_rows)}')

if len(blank_rows) > 0:
    print(f'❌ {len(blank_rows)} blank rows found at index(es): {blank_rows.index.tolist()}')
    print('   Fix: will be removed in the clean step.')
else:
    print('✅ No fully blank rows found.')

In [ ]:
# ── CHECK 5: Duplicate Rows ──────────────────────────────────────────────────
print('=' * 55)
print('CHECK 5 — Duplicate Rows')
print('=' * 55)

dupe_count = df.duplicated().sum()
print(f'Duplicate rows : {dupe_count}')

if dupe_count > 0:
    print(f'❌ {dupe_count} fully duplicate rows found.')
    print('   Fix: will be removed in the clean step.')
    display(df[df.duplicated(keep=False)].head(10))
else:
    print('✅ No duplicate rows found.')

In [ ]:
# ── CHECK 6: Null / Missing Value Summary ────────────────────────────────────
print('=' * 55)
print('CHECK 6 — Null / Missing Value Summary')
print('=' * 55)

null_summary = df.isnull().sum()
null_pct = (df.isnull().mean() * 100).round(1)
null_report = pd.DataFrame({'null_count': null_summary, 'null_%': null_pct})
null_report = null_report[null_report['null_count'] > 0].sort_values('null_count', ascending=False)

if null_report.empty:
    print('✅ No nulls found across all columns.')
else:
    print(f'⚠️  {len(null_report)} column(s) have nulls:')
    display(null_report)

In [ ]:
# ── CHECK 7: Row & Column Count ──────────────────────────────────────────────
print('=' * 55)
print('CHECK 7 — Shape & Data Types')
print('=' * 55)

print(f'Rows    : {df.shape[0]}')
print(f'Columns : {df.shape[1]}')
print()
print('Data types:')
display(df.dtypes.to_frame(name='dtype'))

## 4. Clean & Fix

In [ ]:
# ── Apply all fixes ──────────────────────────────────────────────────────────
df_clean = df.copy()

# Strip BOM from column names if still present
df_clean.columns = df_clean.columns.str.replace('\ufeff', '', regex=False).str.strip()

# Remove fully blank rows
rows_before = len(df_clean)
df_clean = df_clean.dropna(how='all')
rows_after = len(df_clean)
print(f'Blank rows removed  : {rows_before - rows_after}')

# Remove duplicate rows
dupes_before = len(df_clean)
df_clean = df_clean.drop_duplicates()
dupes_after = len(df_clean)
print(f'Duplicate rows removed : {dupes_before - dupes_after}')

# Re-encode string columns to UTF-8 (handles Latin-1 special characters)
for col in df_clean.select_dtypes(include='object').columns:
    df_clean[col] = df_clean[col].apply(
        lambda x: x.encode('utf-8', errors='ignore').decode('utf-8') if isinstance(x, str) else x
    )

print(f'\nFinal shape: {df_clean.shape[0]} rows x {df_clean.shape[1]} columns')
print('✅ Clean dataframe ready.')

## 5. Download Cleaned File

In [ ]:
# ── Save and download ────────────────────────────────────────────────────────
base_name = os.path.splitext(os.path.basename(FILE_PATH))[0]
OUTPUT_PATH = f'{base_name}_clean.csv'

df_clean.to_csv(OUTPUT_PATH, index=False, encoding='utf-8')  # clean UTF-8, no BOM

print(f'✅ Cleaned file saved as: {OUTPUT_PATH}')
print(f'   Encoding : UTF-8 (no BOM)')
print(f'   Rows     : {df_clean.shape[0]}')
print(f'   Columns  : {df_clean.shape[1]}')
print()
display(FileLink(OUTPUT_PATH, result_html_prefix='⬇️ Download: '))

## 6. Summary Report

In [ ]:
# ── Print full summary ───────────────────────────────────────────────────────
print('=' * 55)
print('SUMMARY REPORT')
print('=' * 55)
print(f'File              : {FILE_PATH}')
print(f'BOM detected      : {"❌ Yes — fixed" if BOM_DETECTED else "✅ No"}')
print(f'Encoding          : {detected["encoding"]} → saved as UTF-8')
print(f'Blank rows removed: {rows_before - rows_after}')
print(f'Duplicates removed: {dupes_before - dupes_after}')
print(f'Original shape    : {df.shape[0]} rows x {df.shape[1]} cols')
print(f'Clean shape       : {df_clean.shape[0]} rows x {df_clean.shape[1]} cols')
print(f'Output file       : {OUTPUT_PATH}')
print('=' * 55)